# KLASIFIKASI GENRE GAME BERDASARKAN SINOPSIS
---
**Author:** *Muhammad Wira Widhana [24.55.2717]*

## Library & Load Dataset

library 

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import re
import ast
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer

from sklearn.naive_bayes import MultinomialNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import LinearSVC

from sklearn.metrics import confusion_matrix, classification_report

In [2]:
# pertama kali saja:

nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')

# digunakan di preprocessing
english_stopwords = set(stopwords.words('english'))
stemmer = PorterStemmer()


[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\ShirO_KurO\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\ShirO_KurO\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\ShirO_KurO\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


## Load dan Memeriksa Data (Data Loading & EDA)

Load Dataset

In [3]:
# Load Dataset
df_raw = pd.read_csv('games_fixed.csv')

print("Ukuran awal dataset:", df_raw.shape)
df_raw.head()

Ukuran awal dataset: (122611, 36)


,AppID,Name,Release date,Estimated owners,Peak CCU,Required age,Price,Discount,DLC count,About the game,...,Median playtime two weeks,Average playtime forever,Median playtime forever,Developers,Publishers,Categories,Genres,Tags,Screenshots,Movies
0,10,Counter-Strike,"Nov 1, 2000",10000000 - 20000000,7323,0,1.99,80,0,Play the world's number 1 online action game. ...,...,99,10735,185,['Valve'],['Valve'],"['Multi-player', 'PvP', 'Online PvP', 'Shared/...",['Action'],"{'Action': 5504, 'FPS': 4929, 'Multiplayer': 3...",['https://shared.akamai.steamstatic.com/store_...,[]
1,100,Counter-Strike: Condition Zero,"Mar 1, 2004",10000000 - 20000000,83,0,1.99,80,0,"With its extensive Tour of Duty campaign, a ne...",...,1,431,32,['Valve'],['Valve'],"['Single-player', 'Multi-player', 'Color Alter...",['Action'],"{'Action': 1339, 'FPS': 979, 'Shooter': 723, '...",['https://shared.akamai.steamstatic.com/store_...,[]
2,1000000,ASCENXION,"May 14, 2021",0 - 20000,0,0,1.49,0,1,ASCENXION is a 2D shoot 'em up game where you ...,...,0,0,0,['IndigoBlue Game Studio'],['PsychoFlux Entertainment'],"['Single-player', 'Steam Achievements', 'Parti...","['Action', 'Adventure', 'Indie']","{""Shoot 'Em Up"": 186, 'Metroidvania': 181, 'Bu...",['https://shared.akamai.steamstatic.com/store_...,[]
3,1000010,Crown Trick,"Oct 16, 2020",500000 - 1000000,14,0,4.99,0,2,Crown Trick is a beautifully animated rogue-li...,...,0,702,397,['NEXT Studios'],"['Team17', 'NEXT Studios']","['Single-player', 'Steam Achievements', 'Steam...","['Adventure', 'Indie', 'RPG', 'Strategy']","{'Rogue-like': 275, 'Turn-Based Combat': 258, ...",['https://shared.akamai.steamstatic.com/store_...,[]
4,1000030,"Cook, Serve, Delicious! 3?!","Oct 14, 2020",100000 - 200000,27,0,4.99,75,1,Hit the road in this massive sequel to the mil...,...,0,281,118,['Vertigo Gaming Inc.'],['Vertigo Gaming Inc.'],"['Single-player', 'Multi-player', 'Co-op', 'Sh...","['Action', 'Indie', 'Simulation', 'Strategy']","{'Typing': 227, 'Management': 219, 'Difficult'...",['https://shared.akamai.steamstatic.com/store_...,[]


>Penjelasan singkat: Membaca dataset `games_fixed.csv` yang merupakan dataset dari kaggle 

Exploratory Data Analysis (EDA)

In [10]:
# Exploratory Data Analysis (EDA)

df = df_raw[['Name', 'About the game', 'Genres']].copy()

df.rename(columns={
    'Name': 'title',
    'About the game': 'synopsis',
    'Genres': 'genre_raw'
}, inplace=True)

print("Jumlah data setelah seleksi kolom:", df.shape)
df.sample(5)

Jumlah data setelah seleksi kolom: (122611, 3)


,title,synopsis,genre_raw
116304,Precision Sniping: Competitive,Become a true Sniper. Precision Sniper is an a...,"['Indie', 'Simulation', 'Sports']"
77177,The Lost Robot,"The Lost Robot - For unknown reasons, you foun...",['Indie']
78896,Drinking in the hot spring!,【Important Notice】 - You can change the langua...,"['Adventure', 'Casual', 'Indie', 'Free To Play']"
36082,TAnima Playtest,NaN,[]
57219,Exanimum: The Silent Call,Exanimum: The Silent Call - It is a first-pers...,['Indie']


Cek missing values

In [5]:
# Cek missing values
df.isna().sum()

title           1
synopsis     8448
genre_raw       0
dtype: int64

Hapus baris yang tidak punya sinopsis atau genre

In [11]:
# Hapus baris yang tidak punya sinopsis atau genre
df = df.dropna(subset=['synopsis', 'genre_raw']).reset_index(drop=True)

# Tambahan: hapus genre yang kosong secara konten, seperti [], [''], dll.
df['genre_raw'] = df['genre_raw'].astype(str).str.strip()

# Filter keluar string yang merepresentasikan list kosong
df = df[~df['genre_raw'].isin(['[]', "['']", '[""]', '', 'nan'])].reset_index(drop=True)

print("Jumlah data setelah drop NA dan genre kosong:", df.shape)

Jumlah data setelah drop NA dan genre kosong: (114022, 3)


In [17]:
df.sample(5)

,title,synopsis,genre_raw
104933,Big Tower Tiny Square,Your best friend Pineapple was stolen by Big S...,"['Action', 'Adventure', 'Indie']"
102466,ReallyGoodBattle,"ReallyGoodBattle: Fast-Paced, Action-Arcade, P...","['Action', 'Indie']"
18435,My Last Memories About You,"About the game This short indie game, My Last ...","['Adventure', 'Casual', 'Indie']"
90620,Pimpit,PIMPIT – The Ultimate 3D Car Tuning &amp; Cust...,"['Indie', 'Simulation', 'Free To Play']"
47010,Angels vs Angels,Angels vs Angels A fast-paced multiplayer acti...,"['Action', 'Adventure', 'Indie', 'Massively Mu..."


Normalisasi Genre

In [18]:
target_genres = {'action', 'adventure', 'rpg', 'simulation', 'casual', 'indie'}

def parse_genre_list(genre_str):
    """
    Parse string genre seperti "['Action', 'Indie']" menjadi list python ['action', 'indie'].
    Tidak melakukan penghapusan manual karakter [ ] dengan replace,
    melainkan parsing aman menggunakan ast.literal_eval.
    """
    if pd.isna(genre_str):
        return []
    
    # Jika sudah list, kembalikan langsung
    if isinstance(genre_str, list):
        raw_list = genre_str
    else:
        text = str(genre_str).strip()
        try:
            # Coba parse sebagai literal Python
            raw_list = ast.literal_eval(text)
        except (ValueError, SyntaxError):
            # Jika gagal, fallback split sederhana dengan koma
            raw_list = [g.strip() for g in text.split(',')]
    
    # Normalisasi ke huruf kecil tanpa spasi ekstra
    cleaned = [g.strip().lower() for g in raw_list if isinstance(g, str)]
    return cleaned

# Terapkan parsing ke seluruh kolom genre_raw
df['genre_list'] = df['genre_raw'].apply(parse_genre_list)

# Ambil hanya genre yang termasuk subset target
df['genre_filtered'] = df['genre_list'].apply(
    lambda lst: [g for g in lst if g in target_genres]
)

# Hitung panjang list hasil filter
df['genre_count'] = df['genre_filtered'].apply(len)

print("Distribusi jumlah genre target per game:")
print(df['genre_count'].value_counts().sort_index())

Distribusi jumlah genre target per game:
genre_count
0     3968
1    22223
2    39684
3    31891
4    12428
5     3055
6      773
Name: count, dtype: int64


In [19]:
df.sample(5)

,title,synopsis,genre_raw,genre_list,genre_filtered,genre_count
111432,Reflections of Life: Tree of Dreams Collector'...,"From GrandMA Studios, creators of Whispered Se...","['Adventure', 'Casual']","[adventure, casual]","[adventure, casual]",2
71576,Runes of Vespera,Runes of Vespera is a roguelite FPS survival g...,"['Action', 'Indie', 'RPG', 'Strategy']","[action, indie, rpg, strategy]","[action, indie, rpg]",3
30274,A Fungus In My Garden,Would you dare face ‘something’ way bigger tha...,"['Action', 'Indie']","[action, indie]","[action, indie]",2
18505,Absurbia: A Trashy Satire of Suburban Outcries,"For whatever reason, you might have stumbled t...","['Casual', 'Indie']","[casual, indie]","[casual, indie]",2
64247,Ultrakanoid,Ultrakanoid is an exciting game that will allo...,"['Casual', 'Indie']","[casual, indie]","[casual, indie]",2


In [24]:
# Untuk menjaga single-label classification sesuai proposal,
# hanya ambil data yang memiliki tepat satu genre target
df_single = df[df['genre_count'] == 1].copy().reset_index(drop=True)

# Ambil genre utama
df_single['genre_main'] = df_single['genre_filtered'].str[0]

print("Jumlah data setelah filter subset genre dan single-label:", df_single.shape)
df_single[['title','synopsis' , 'genre_raw', 'genre_list', 'genre_filtered', 'genre_main']].head(10)

Jumlah data setelah filter subset genre dan single-label: (22223, 7)


,title,synopsis,genre_raw,genre_list,genre_filtered,genre_main
0,Counter-Strike,Play the world's number 1 online action game. ...,['Action'],[action],[action],action
1,Counter-Strike: Condition Zero,"With its extensive Tour of Duty campaign, a ne...",['Action'],[action],[action],action
2,Hellish Quart,Hellish Quart is a local-only (two players pla...,"['Action', 'Early Access']","[action, early access]",[action],action
3,WRATH: Aeon of Ruin,You are Outlander. Once adrift upon the Ageles...,['Action'],[action],[action],action
4,The ScreaMaze,Gameplay: Take your mouse and try to reach the...,['Indie'],[indie],[indie],indie
5,RogueCraft Squadron,RogueCraft Squadron is a fast paced real-time ...,"['Indie', 'Strategy']","[indie, strategy]",[indie],indie
6,Glorious Companions,Glorious Companions is a turn-based tactical R...,"['RPG', 'Strategy']","[rpg, strategy]",[rpg],rpg
7,Angry Birds VR: Isle of Pigs,"Join Red, Chuck, Bomb and the Blues to save th...",['Casual'],[casual],[casual],casual
8,VR Flight Simulator New York - Cessna,VR Flight Simulator New York - Cessna is the u...,['Simulation'],[simulation],[simulation],simulation
9,Kebab Chefs! - Restaurant Simulator,◆ You wouldn't say no for a little help with t...,"['Simulation', 'Early Access']","[simulation, early access]",[simulation],simulation


sebaran genre

In [25]:
df_single['genre_main'].value_counts()

genre_main
indie         5780
casual        4791
action        4485
simulation    2768
adventure     2581
rpg           1818
Name: count, dtype: int64

## Filter Bahasa Inggris & Text Cleaning

## PRE-Prosessing


preprocessing mencakup case folding, tokenizing, stopword removal, dan stemming, dan batasan masalah perubahan kombinasi fitur preprocessing.

Skenario:

- S1: Case folding + tokenizing + stopword removal

- S2: Case folding + tokenizing + stemming

- S3: Case folding + tokenizing + stopword removal + stemming

## Modeling & Training

Spliting model sebelum train spliting data menjadi 80 20


`test_size=0.2` 20% data untuk test

Sisanya 80% untuk model train

Model Naive Bayes

In [ ]:
# NAIVE BAYES

nb_clf = MultinomialNB()
nb_clf.fit(X_train, y_train)

y_pred_nb = nb_clf.predict(X_test)

Model K-Nearest Neighbors

In [ ]:
# K-NEAREST NEIGHBORS

# K bisa disesuaikan, misal k=19 mengacu Putra et al. (2024) untuk kasus anime
knn_clf = KNeighborsClassifier(n_neighbors=19, metric='cosine')
knn_clf.fit(X_train, y_train)

y_pred_knn = knn_clf.predict(X_test)

Model Support Vector Machine

In [ ]:
# SUPPORT VECTOR MACHINE 

svm_clf = LinearSVC()
svm_clf.fit(X_train, y_train)

y_pred_svm = svm_clf.predict(X_test)

## Evaluasi Model


## Implementasi